imports and load in data into df

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path("../data/raw/nfl-big-data-bowl-2027")

df = pd.read_csv(RAW / "player_career_successes.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   nfl_id                      510 non-null    int64
 1   career_offensive_snaps      510 non-null    int64
 2   career_defensive_snaps      510 non-null    int64
 3   career_special_teams_snaps  510 non-null    int64
 4   career_games_active         510 non-null    int64
 5   career_games_started        510 non-null    int64
 6   ap_all_pro_1st_team         510 non-null    int64
 7   ap_all_pro_2nd_team         510 non-null    int64
 8   pro_bowl_original_ballot    510 non-null    int64
dtypes: int64(9)
memory usage: 36.0 KB


Use pro bowl original ballot to consider if player is 'good'

establish features and log data to get rid of skewness from large dataset

In [8]:
df['is_good'] = (df['pro_bowl_original_ballot'] >= 1).astype(int)

FEATURES = [
    'career_offensive_snaps',
    'career_defensive_snaps',
    'career_special_teams_snaps',
    'career_games_active',
    'career_games_started'
]

X = np.log1p(df[FEATURES].to_numpy(dtype=float))
y = df['is_good'].to_numpy(dtype=float)

print("share labeled good:", y.mean())

share labeled good: 0.021568627450980392


Standardize the data and split into train/test sets randomly

In [10]:
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
split = int(len(X) * 0.8)
train, test = idx[:split], idx[split:]

mu, sd = X[train].mean(axis=0), X[train].std(axis=0)
sd[sd == 0] = 1

X_train = (X[train] - mu) / sd
X_test = (X[test] - mu) / sd
y_train, y_test = y[train], y[test]

Perform gradient descent

In [11]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1

for epoch in range(2000):
    p = sigmoid(X_train @ w + b)
    err = p - y_train

    dw = X_train.T @ err / len(y_train)
    db = err.mean()

    w -= lr * dw
    b -= lr * db

    if epoch % 200 == 0:
        loss = -np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))
        print(f"epoch {epoch:4d}: loss={loss:.4f}")

print(dict(zip(FEATURES, w.round(3))), "bias:", round(b,3))

epoch    0: loss=0.6931
epoch  200: loss=0.1201
epoch  400: loss=0.0980
epoch  600: loss=0.0900
epoch  800: loss=0.0856
epoch 1000: loss=0.0828
epoch 1200: loss=0.0808
epoch 1400: loss=0.0793
epoch 1600: loss=0.0782
epoch 1800: loss=0.0772
{'career_offensive_snaps': np.float64(-0.12), 'career_defensive_snaps': np.float64(0.36), 'career_special_teams_snaps': np.float64(-0.439), 'career_games_active': np.float64(0.375), 'career_games_started': np.float64(1.497)} bias: -4.667


/var/folders/nr/wjfq_svj5_vdm27q_1b1_f8m0000gn/T/ipykernel_75560/3341783518.py:9: RuntimeWarning: divide by zero encountered in matmul
  p = sigmoid(X_train @ w + b)
/var/folders/nr/wjfq_svj5_vdm27q_1b1_f8m0000gn/T/ipykernel_75560/3341783518.py:9: RuntimeWarning: overflow encountered in matmul
  p = sigmoid(X_train @ w + b)
/var/folders/nr/wjfq_svj5_vdm27q_1b1_f8m0000gn/T/ipykernel_75560/3341783518.py:9: RuntimeWarning: invalid value encountered in matmul
  p = sigmoid(X_train @ w + b)


In [ ]:
p_test = sigmoid(X_test @ w + b)
pred = (p_test >= 0.5).astype(int)

tp = ((pred == 1) and (y_test == 1)).sum()
fp = ((pred == 1) and (y_test == 0)).sum()
fn = ((pred == 0) and (y_test == 1)).sum()
